Conexao DB

In [ ]:
pip install oracledb

In [ ]:
import os
import oracledb
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

# -----------------------------
# 1. Connection parameters
# -----------------------------
username = os.environ.get("ORACLE_USERNAME", "")
password = os.environ.get("ORACLE_PASSWORD", "")
host = os.environ.get("ORACLE_HOST", "")
port = int(os.environ.get("ORACLE_PORT", "1521"))
service_name = os.environ.get("ORACLE_SERVICE_NAME", "CDE")

if not all([username, password, host, service_name]):
    raise ValueError("Set ORACLE_USERNAME, ORACLE_PASSWORD, ORACLE_HOST and ORACLE_SERVICE_NAME before running")

connection_string = (
    f"oracle+oracledb://{username}:{password}@{host}:{port}/?service_name={service_name}"
)
engine = create_engine(connection_string)

FETCH DB PARA DF

In [ ]:
# Colunas a incluir na consulta (deixe vazio [] para selecionar todas)
columns_to_select = [
    # Descomentar e adicionar as colunas desejadas:
    'ENTITY_ID', # ID
    'STATE', # ESTADO
    'COUPON_CODE', 
    'STORE_ID', # PAIS LOJA
    'CUSTOMER_ID', 
    'BASE_DISCOUNT_INVOICED', # DESCONTO BASE FATURADO
    'BASE_DISCOUNT_REFUNDED', # DESCONTO BASE DEVOLVIDO
    'BASE_GRAND_TOTAL', # TOTAL BASE PRODUTOS 
    'BASE_SHIPPING_INVOICED', # FRETE BASE FATURADO
    'BASE_SUBTOTAL_INVOICED', # GRANDTOTAL + SHIPPING - SE STATUS == "closed" foi refunded
    'BASE_TO_ORDER_RATE', # COTACAO MOEDA = ao STORE_TO_ORDER_RATE
    'BASE_TOTAL_INVOICED_COST', # CUSTO TOTAL BASE FATURADO
    'TOTAL_QTY_ORDERED', # QTD TOTAL PEDIDA
    'CUSTOMER_IS_GUEST',
    'CUSTOMER_GROUP_ID', # EXCEL:  1- "Registered Client",   7 - "Team",  9 - "Friends" ,2 - "", 8-"", os restantes "Not Registered"
    'SHIPPING_ADDRESS_ID',
    'WEIGHT',
    'INCREMENT_ID', # EXCEL: campo usado para excluir marketplaces
    'ORDER_CURRENCY_CODE', # MOEDA DO PEDIDO
    'STORE_NAME', # NOME DA LOJA
    'CREATED_AT',
    'UPDATED_AT', 
    'CUSTOMER_GENDER', # so existem 135 com valor 1 ou 2, o resto é null
    'TOTAL_ITEM_COUNT', # VER SE É IGUAL AO TOTAL_QTY_ORDERED
    'COUPON_RULE_NAME',
    'BASE_SUBTOTAL_WITH_DISCOUNT', # ver se é igual a BASE_SUBTOTAL_INVOICED
    'ANALYTICS', # DÁ PARA EXTRAIR O DEVICE DAQUI MOBILE VS WEB
    # 'UPDATED_AT',
]

# Query para obter dados da tabela SALES_FLAT_ORDER
table_owner = "C2B"
table_name = "SALES_FLAT_ORDER"

print(f"Querying table: {table_owner}.{table_name}")

# Construir query com colunas específicas ou todas
if columns_to_select:
    columns_str = ', '.join([f'"{col}"' for col in columns_to_select])
    query = f'SELECT {columns_str} FROM "{table_owner}"."{table_name}"'
else:
    query = f'SELECT * FROM "{table_owner}"."{table_name}"'

print(f"Query: {query[:100]}...")

# Executar query
df = pd.read_sql(query, engine)

print(f"Total rows retrieved: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumn names:\n{list(df.columns)}")

df.head(5)

df.to_csv("CSVTables/SALES_FLAT_ORDER.csv", index=False, encoding='utf-8')
print("Data exported to CSVTables/SALES_FLAT_ORDER.csv")

COPIAR AS RESTANTES

In [ ]:
#table_names = ["CUSTOMER_DATA","EAV_ATTRIBUTE_OPTION_VALUE","SALES_FLAT_ORDER_ITEM","SALES_FLAT_ORDER_ADDRESS"]
table_name=["SALES_FLAT_ORDER"]
table_owner = "C2B"
output_dir = os.getcwd()
for table_name in table_name:
    synonym_name = table_name
    print(f"Processing table: {synonym_name}")
    
    try:
        query = f'SELECT * FROM "{table_owner}"."{table_name}"'
        table_df = pd.read_sql(query, engine)
        
        csv_filename = os.path.join(output_dir, f"{synonym_name}.csv")
        table_df.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"Data exported to {csv_filename}")
    except Exception as e:
        print(f"Error processing table {synonym_name}: {e}")
